In [1]:
%%capture
!pip install torch>=1.10.0
!pip install datasets
!pip install tqdm

In [2]:
import sacrebleu
import torch
from huggingface_hub import list_repo_refs, notebook_login
from datasets import load_dataset, get_dataset_split_names, concatenate_datasets, DatasetDict
from tqdm import tqdm
import os

import copy

import numpy as np

In [3]:
COCO_SUBSET_ID = "umairhassan02/coco-captions-subset-59k"
COCO_TRANSLATED_ID = "umairhassan02/coco-captions-subset-refined"

In [4]:
coco_subset = load_dataset(COCO_SUBSET_ID)

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/58549 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [5]:
cococ_translated_dataset = load_dataset(COCO_TRANSLATED_ID)

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

In [9]:
id2translated = {}
for split in ["train", "validation"]:
    for row in tqdm(cococ_translated_dataset[split]):
        if row["image_id"] not in id2translated:
            id2translated[row["image_id"]] = []
        id2translated[row["image_id"]].append(row["translated_captions"])

100% 292907/292907 [04:10<00:00, 1170.23it/s]
100% 25014/25014 [00:21<00:00, 1187.65it/s]


In [10]:
len(id2translated)

63549

In [13]:
coco_subset['train'][3822]

{'image_id': 134386,
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=480x640>,
 'file_name': '000000134386.jpg',
 'coco_url': 'http://images.cocodataset.org/train2017/000000134386.jpg',
 'height': 640,
 'width': 480,
 'date_captured': '2013-11-18 20:21:13',
 'flickr_url': 'http://farm3.staticflickr.com/2350/2243134195_35f1f8ef5c_z.jpg',
 'license_id': 4,
 'license': {'url': 'http://creativecommons.org/licenses/by/2.0/',
  'license_id': 4,
  'name': 'Attribution License'},
 'annotations': {'annotation_id': [34090, 34273, 41098, 42805, 46963],
  'image_id': [134386, 134386, 134386, 134386, 134386],
  'caption': ['A lady in a shop with a piece of cake with lit candles.',
   'A girl in a shop holding a plate with a dessert and candles.',
   'A woman sitting in a store holding a cake with lit candles.',
   'Smiling lady appears to be staying stop while holding a cake with candles.',
   'A cute girl holding a plate with a cake and lit candles.']}}

In [14]:
id2translated[134386]

['ایک دکان میں ایک خاتون کے ساتھ کیک کا ایک ٹکڑا روشن موم بتیوں کے ساتھ.',
 'ایک دکان میں ایک لڑکی ایک میٹھی اور موم بتیوں کے ساتھ ایک پلیٹ پکڑنے .',
 'ایک عورت ایک دکان میں بیٹھی ہے جس میں روشن موم بتیوں کے ساتھ کیک پکڑا ہوا ہے۔',
 'مسکراتے ہوئے خاتون موم بتیوں کے ساتھ کیک تھامے ہوئے رکنے لگتی ہے۔',
 'ایک پیاری لڑکی ایک کیک اور روشن موم بتیاں کے ساتھ ایک پلیٹ پکڑ.']

In [19]:
subset = copy.deepcopy(coco_subset)

In [25]:
for split in ["train", "validation"]:
    new_annotations = []
    for ann in tqdm(subset[split]["annotations"]):
        img_ids = ann["image_id"]
        assert all(i == img_ids[0] for i in img_ids)
        ann["translated_captions"] = id2translated[img_ids[0]]
        new_annotations.append(ann)
    # Replace annotations in-place
    subset[split] = subset[split].remove_columns("annotations").add_column("annotations", new_annotations)

100% 58549/58549 [00:00<00:00, 1358952.92it/s]
100% 5000/5000 [00:00<00:00, 1224113.94it/s]


In [31]:
subset['validation'][3829]

{'image_id': 68078,
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=360x640>,
 'file_name': '000000068078.jpg',
 'coco_url': 'http://images.cocodataset.org/val2017/000000068078.jpg',
 'height': 640,
 'width': 360,
 'date_captured': '2013-11-15 05:58:05',
 'flickr_url': 'http://farm5.staticflickr.com/4017/4331592297_aa22c03ebe_z.jpg',
 'license_id': 1,
 'license': {'url': 'http://creativecommons.org/licenses/by-nc-sa/2.0/',
  'license_id': 1,
  'name': 'Attribution-NonCommercial-ShareAlike License'},
 'annotations': {'annotation_id': [748284, 748569, 750945, 751614, 751976],
  'caption': ['A dog in a small bathroom with an orange shower curtain.',
   'A dog that is standing on a rug in a bathroom.',
   'A small dug stands on a white rug in the bathroom.',
   'The dog was standing in the bathroom near the sink and toilet.',
   'A puppy sitting on the ground in a bathroom'],
  'image_id': [68078, 68078, 68078, 68078, 68078],
  'translated_captions': ['ایک چھوٹے سے باتھ رو

In [32]:
coco_subset['validation'][3829]

{'image_id': 68078,
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=360x640>,
 'file_name': '000000068078.jpg',
 'coco_url': 'http://images.cocodataset.org/val2017/000000068078.jpg',
 'height': 640,
 'width': 360,
 'date_captured': '2013-11-15 05:58:05',
 'flickr_url': 'http://farm5.staticflickr.com/4017/4331592297_aa22c03ebe_z.jpg',
 'license_id': 1,
 'license': {'url': 'http://creativecommons.org/licenses/by-nc-sa/2.0/',
  'license_id': 1,
  'name': 'Attribution-NonCommercial-ShareAlike License'},
 'annotations': {'annotation_id': [748284, 748569, 750945, 751614, 751976],
  'image_id': [68078, 68078, 68078, 68078, 68078],
  'caption': ['A dog in a small bathroom with an orange shower curtain.',
   'A dog that is standing on a rug in a bathroom.',
   'A small dug stands on a white rug in the bathroom.',
   'The dog was standing in the bathroom near the sink and toilet.',
   'A puppy sitting on the ground in a bathroom']}}

In [34]:
# let's upload and we are done yay!
FINAL_DATASET_ID = "umairhassan02/urdu-translated-coco-captions-subset"
subset.push_to_hub(FINAL_DATASET_ID)

Uploading the dataset shards:   0%|          | 0/20 [00:00<?, ?it/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/30 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

...

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/umairhassan02/urdu-translated-coco-captions-subset/commit/8270350b9a2e8988cfa64897df2b4623138ea590', commit_message='Upload dataset', commit_description='', oid='8270350b9a2e8988cfa64897df2b4623138ea590', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/umairhassan02/urdu-translated-coco-captions-subset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='umairhassan02/urdu-translated-coco-captions-subset'), pr_revision=None, pr_num=None)